# Phase 2 — Seed Nanobody Generation

This notebook walks through the seed generation pipeline for the
NanoLFA-Design project. We prepare the starting pool of nanobody
candidates that enter the iterative design loop in Phase 3.

**What this notebook covers:**
1. Load and curate VHH germline scaffolds (bundled library)
2. Validate VHH hallmark residues and canonical disulfide
3. Annotate CDR/framework regions (IMGT scheme)
4. Cluster scaffolds by framework identity to ensure diversity
5. Screen scaffolds with ESMFold for fold quality
6. Analyze per-region pLDDT (framework vs. CDR confidence)
7. Select the seed scaffold library for downstream design

**Prerequisites:**
```bash
conda activate nanolfa
pip install -e .
# ESMFold requires: pip install fair-esm
# GPU recommended for ESMFold (runs on CPU but ~50x slower)
```

**Note:** If ESMFold is not available (no GPU, missing `fair-esm`),
the scaffold curation sections (1–4) still work. The ESMFold screening
sections (5–6) will be skipped with a warning.

In [ ]:
import logging
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from nanolfa.utils.sequence import (
    annotate_regions,
    validate_vhh,
    load_bundled_germlines,
    curate_scaffold_library,
    cluster_sequences,
    IMGT_REGIONS,
    VHH_HALLMARKS,
)

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

OUTPUT_DIR = Path('../data/templates/germline_vhh')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Check ESMFold availability
ESMFOLD_AVAILABLE = False
try:
    import esm
    import torch
    ESMFOLD_AVAILABLE = torch.cuda.is_available()
    if not ESMFOLD_AVAILABLE:
        print('ESMFold available but no GPU detected — screening will be slow.')
        print('Set ESMFOLD_AVAILABLE = True to run on CPU anyway.')
    else:
        print(f'ESMFold ready (GPU: {torch.cuda.get_device_name(0)})')
except ImportError:
    print('ESMFold not available (fair-esm not installed). Skipping fold screening.')

print('Setup complete.')

## 1. Load Germline Scaffold Library

The bundled library contains 10 representative VHH sequences from
alpaca (*Vicugna pacos*) and dromedary (*Camelus dromedarius*) —
the two best-characterized camelid species for nanobody engineering.

Each scaffold is a full-length VHH sequence including CDR loops.
In Phase 3, we'll redesign the CDR loops (particularly CDR3) while
keeping the framework frozen.

In [ ]:
# Load all bundled scaffolds
all_scaffolds = load_bundled_germlines()
print(f'Loaded {len(all_scaffolds)} scaffolds\n')

# Summary table
print(f'{"Name":<30} {"Species":<22} {"Len":>4} {"V gene":<12} {"J gene":<8}')
print('-' * 80)
for s in all_scaffolds:
    print(f'{s.name:<30} {s.species:<22} {len(s.sequence):>4} {s.v_gene:<12} {s.j_gene:<8}')

## 2. VHH Hallmark Validation

VHH nanobodies are distinguished from conventional VH domains by specific
amino acid substitutions at IMGT positions 37, 44, 45, and 47. These
"hallmark" residues compensate for the absence of the VL partner chain
by increasing hydrophilicity on the former VH-VL interface.

We also verify the canonical disulfide bond (Cys23–Cys104) which is
essential for fold stability.

In [ ]:
# Validate all scaffolds
print(f'{"Name":<30} {"VHH Score":>9} {"Disulfide":>9} {"Hallmarks":>10} {"Warnings"}')
print('-' * 90)
for s in all_scaffolds:
    v = s.validation
    if v is None:
        continue
    disulfide = 'Yes' if v.has_canonical_disulfide else 'NO'
    hallmarks = f'{v.hallmark_matches}/{v.hallmark_total}'
    warn_str = '; '.join(v.warnings[:2]) if v.warnings else 'none'
    print(f'{s.name:<30} {v.vhh_score:>9.2f} {disulfide:>9} {hallmarks:>10} {warn_str}')

In [ ]:
# VHH score distribution
scores = [s.validation.vhh_score for s in all_scaffolds if s.validation]

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(
    [s.name for s in all_scaffolds],
    scores,
    color=['#4CAF50' if sc >= 0.6 else '#FF5722' for sc in scores],
    height=0.6,
)
ax.axvline(x=0.6, color='black', linestyle='--', alpha=0.5, label='Threshold (0.6)')
ax.set_xlabel('VHH Likeness Score', size=12)
ax.set_title('VHH Validation: Hallmark Residues + Canonical Disulfide', size=13)
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'vhh_validation_scores.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. CDR Region Annotation

Annotate each scaffold with IMGT CDR/framework boundaries.
CDR3 length is particularly important — it's the primary specificity
determinant and we need diverse starting lengths.

In [ ]:
# CDR lengths for each scaffold
print(f'{"Name":<30} {"CDR1":>5} {"CDR2":>5} {"CDR3":>5} {"Total":>6}')
print('-' * 55)
for s in all_scaffolds:
    r = s.regions
    if r is None:
        continue
    total_len = len(r.cdr1) + len(r.cdr2) + len(r.cdr3)
    print(f'{s.name:<30} {len(r.cdr1):>5} {len(r.cdr2):>5} {len(r.cdr3):>5} {total_len:>6}')

In [ ]:
# CDR3 length distribution
cdr3_lengths = [s.regions.cdr3_length for s in all_scaffolds if s.regions]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(cdr3_lengths, bins=range(min(cdr3_lengths), max(cdr3_lengths) + 2),
        color='#2196F3', edgecolor='white', alpha=0.8)
ax.set_xlabel('CDR3 Length (residues)', size=12)
ax.set_ylabel('Count', size=12)
ax.set_title('CDR3 Length Distribution in Scaffold Library', size=13)

# Overlay target ranges from configs
ax.axvspan(12, 20, alpha=0.1, color='red', label='PdG target (12-20)')
ax.axvspan(10, 18, alpha=0.1, color='blue', label='E3G target (10-18)')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'cdr3_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Framework Clustering

Cluster scaffolds by framework sequence identity to ensure we don't
over-represent any single germline family. The curate function filters
by VHH score, clusters at 90% identity, and picks one representative
per cluster.

In [ ]:
curated = curate_scaffold_library(
    all_scaffolds,
    min_vhh_score=0.6,
    cluster_identity=0.90,
    min_clusters=5,
)

print(f'\nCurated library: {len(curated)} scaffolds\n')
print(f'{"Name":<30} {"Cluster":>7} {"VHH":>5} {"CDR3":>5}')
print('-' * 50)
for s in curated:
    cdr3_len = s.regions.cdr3_length if s.regions else 0
    vhh = s.validation.vhh_score if s.validation else 0
    print(f'{s.name:<30} {s.cluster_id:>7} {vhh:>5.2f} {cdr3_len:>5}')

## 5. ESMFold Structure Screening

ESMFold predicts protein structure from sequence alone (no MSA needed)
in seconds. We use it to verify that each scaffold folds into a
well-defined immunoglobulin domain before entering the design loop.

Key metrics:
- **Mean pLDDT > 70**: well-folded domain
- **Framework pLDDT > 80**: stable framework (good)
- **CDR pLDDT**: lower is expected (loops are flexible), but very low
  (< 40) suggests the loop conformation is poorly defined
- **CDR confidence ratio** (CDR pLDDT / framework pLDDT): closer to 1.0
  suggests rigid CDR loops (faster kon in LFA)

In [ ]:
if not ESMFOLD_AVAILABLE:
    print('ESMFold not available — skipping structure screening.')
    print('To enable: install fair-esm and ensure a GPU is available.')
    print('\nContinuing with sequence-based curation only.')
    esmfold_results = None
else:
    from nanolfa.models.esmfold import ESMFoldPrescreen
    from omegaconf import OmegaConf

    esm_config = OmegaConf.create({
        'enabled': True,
        'min_plddt_threshold': 60,
        'top_k_to_full_af': 100,
    })

    screener = ESMFoldPrescreen(esm_config)

    sequences = [(s.name, s.sequence) for s in curated]
    pdb_dir = OUTPUT_DIR / 'predicted_structures'
    esmfold_results = screener.screen_scaffolds(sequences, save_dir=pdb_dir)

    print(f'\n{"Name":<30} {"pLDDT":>6} {"FW":>6} {"CDR":>6} {"CDR3":>6} {"Ratio":>6}')
    print('-' * 65)
    for r in esmfold_results:
        print(
            f'{r.candidate_id:<30} {r.mean_plddt:>6.1f} '
            f'{r.mean_framework_plddt:>6.1f} {r.mean_cdr_plddt:>6.1f} '
            f'{r.cdr3_plddt:>6.1f} {r.cdr_confidence_ratio:>6.2f}'
        )

In [ ]:
if esmfold_results:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Per-region pLDDT heatmap
    ax = axes[0]
    regions = ['FR1', 'CDR1', 'FR2', 'CDR2', 'FR3', 'CDR3', 'FR4']
    names = [r.candidate_id for r in esmfold_results]
    data = np.array([
        [r.region_plddt.get(reg, 0) for reg in regions]
        for r in esmfold_results
    ])
    im = ax.imshow(data, cmap='RdYlGn', aspect='auto', vmin=40, vmax=95)
    ax.set_xticks(range(len(regions)))
    ax.set_xticklabels(regions, rotation=45)
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels([n[:20] for n in names], size=8)
    ax.set_title('Per-Region pLDDT', size=13)
    plt.colorbar(im, ax=ax, label='pLDDT')

    # CDR confidence ratio
    ax = axes[1]
    ratios = [r.cdr_confidence_ratio for r in esmfold_results]
    colors = ['#4CAF50' if ratio > 0.8 else '#FF9800' if ratio > 0.6 else '#F44336'
              for ratio in ratios]
    ax.barh(names, ratios, color=colors, height=0.6)
    ax.axvline(x=0.8, color='green', linestyle='--', alpha=0.5, label='Good (>0.8)')
    ax.axvline(x=0.6, color='orange', linestyle='--', alpha=0.5, label='Marginal (0.6)')
    ax.set_xlabel('CDR / Framework pLDDT Ratio', size=12)
    ax.set_title('CDR Confidence Ratio', size=13)
    ax.legend(loc='lower right')
    ax.invert_yaxis()

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'esmfold_screening.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('(Skipped — ESMFold not available)')

## 6. Export Seed Library

Save the final curated scaffold library as FASTA and JSON for
use in Phase 3 (iterative design loop).

In [ ]:
# Save curated library as FASTA
fasta_path = OUTPUT_DIR / 'scaffolds.fasta'
with open(fasta_path, 'w') as f:
    for s in curated:
        vhh = s.validation.vhh_score if s.validation else 0
        cdr3 = s.regions.cdr3_length if s.regions else 0
        f.write(f'>{s.name} vhh_score={vhh:.2f} cdr3_len={cdr3}\n')
        f.write(f'{s.sequence}\n')

print(f'Saved {len(curated)} scaffolds to {fasta_path}')
print(f'\nThese scaffolds are ready for Phase 3: CDR redesign via ProteinMPNN.')

## Summary

**What we built:**
- A curated library of diverse VHH framework scaffolds
- Each scaffold is validated for VHH hallmarks, canonical disulfide,
  and (optionally) fold quality via ESMFold
- Scaffolds are clustered to ensure framework diversity

**What happens next (Phase 3):**
1. For each scaffold, RFdiffusion generates diverse CDR3 backbone geometries
2. ProteinMPNN designs CDR sequences on those backbones
3. ESMFold prescreens the designed sequences (fast triage)
4. AlphaFold-Multimer/AF3 predicts complexes with PdG/E3G
5. The iterative loop begins: predict → score → diversify → re-rank

---

**Next:** Phase 3 — Iterative Design Loop (notebook `03_design_loop.ipynb`)